### **Enviornment**

In [1]:
# Setting up the local imports and env variables
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ['LANGCHAIN_TRACING_V2'] = os.getenv("LANGCHAIN_TRACING_V2")
os.environ['LANGCHAIN_ENDPOINT'] = os.getenv("LANGCHAIN_ENDPOINT")
os.environ['LANGCHAIN_API_KEY'] = os.getenv("LANGCHAIN_API_KEY")
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")

## **1. Chunk Optimization** 

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings,ChatOllama
from langchain_community.vectorstores import Chroma
from langchain_experimental.text_splitter import SemanticChunker

# Embed
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

# Load one large document
loader = WebBaseLoader(
    "https://lilianweng.github.io/posts/2023-06-23-agent/"
)

docs = loader.load()

text_splitter = SemanticChunker(              # This is the main logic of the code...one simple line make it semantic chunking method
    embeddings,
    breakpoint_threshold_type="percentile"
)

chunks = text_splitter.split_documents(docs)

print(f"Original documents: {len(docs)}")
print(f"Chunks created: {len(chunks)}")


# Store Chunks in vector Database 
vectorstore = Chroma.from_documents(documents = chunks , embedding= embeddings , collection_name = "chunk-demo")

# Query
query = "How does memory work in AI agents?"

results = vectorstore.similarity_search(query , k = 2)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_5704\3306860198.py:5: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


Original documents: 1
Chunks created: 21


# **2. Multi-representation Indexing**

In [5]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
docs = loader.load()

loader = WebBaseLoader("https://lilianweng.github.io/posts/2024-02-05-human-data-quality/")
docs.extend(loader.load())

In [7]:
import uuid 
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama , OllamaEmbeddings

llm = ChatOllama(model="llama3.2:3b", temperature=0.2)

chain = (
    {"doc" : lambda x : x.page_content} | 
    ChatPromptTemplate.from_template("Summarize the following document:\n\n{doc}") |
    llm | StrOutputParser()
)

summaries = chain.batch(docs , {"max_concurrency": 5})

In [16]:
from langchain_core.stores import InMemoryByteStore,InMemoryStore
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_classic.retrievers import MultiVectorRetriever

# Vector Store 
vectorstore = Chroma(collection_name="summaries" , embedding_function=embeddings)

# Correct doc store
store = InMemoryStore()

id_key = "doc_id"

# Retriever
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    docstore=store,
    id_key=id_key,
)

# Create IDs
doc_ids = [str(uuid.uuid4()) for _ in docs]

# Store summaries in vector DB
summary_docs = [
    Document(
        page_content=summaries[i],
        metadata={id_key: doc_ids[i]}
    )
    for i in range(len(summaries))
]

retriever.vectorstore.add_documents(summary_docs)

# Store original documents
retriever.docstore.mset(
    list(zip(doc_ids, docs))
)

In [14]:
query = "Memory in agents"
sub_docs = vectorstore.similarity_search(query,k=1)
sub_docs[0]

Document(metadata={'doc_id': 'eafc55a4-0c24-4947-98db-88dee617319e'}, page_content='I\'ll provide a response to the prompt "LLM-powered Autonomous Agents" by Lilian Weng.\n\n**Introduction**\n\nAutonomous agents powered by large language models (LLMs) have gained significant attention in recent years. These agents can learn from vast amounts of data, reason about complex problems, and interact with their environment in a human-like way. In this article, we will explore the challenges and limitations of building LLM-centered autonomous agents.\n\n**Challenges**\n\n1. **Finite context length**: The restricted context capacity limits the inclusion of historical information, detailed instructions, API call context, and responses. This limitation makes it challenging to learn from past mistakes and adapt to new situations.\n2. **Long-term planning and task decomposition**: Planning over a lengthy history and effectively exploring the solution space remain challenging. LLMs struggle to adjus

In [18]:
retrieved_docs = retriever.invoke(query)
retrieved_docs[0].page_content[:5000]

'\n\n\n\n\n\nLLM Powered Autonomous Agents | Lil\'Log\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nLil\'Log\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n|\n\n\n\n\n\n\nPosts\n\n\n\n\nArchive\n\n\n\n\nSearch\n\n\n\n\nTags\n\n\n\n\nFAQ\n\n\n\n\n\n\n\n\n\n      LLM Powered Autonomous Agents\n    \nDate: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng\n\n\n \n\n\nTable of Contents\n\n\n\nAgent System Overview\n\nComponent One: Planning\n\nTask Decomposition\n\nSelf-Reflection\n\n\nComponent Two: Memory\n\nTypes of Memory\n\nMaximum Inner Product Search (MIPS)\n\n\nComponent Three: Tool Use\n\nCase Studies\n\nScientific Discovery Agent\n\nGenerative Agents Simulation\n\nProof-of-Concept Examples\n\n\nChallenges\n\nCitation\n\nReferences\n\n\n\n\n\nBuilding agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The p

# **3. RAPTOR**

In [ ]:
import os
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

LLM_MODEL = "gpt-4o-mini"

class Node:

    def __init__(self, text, children=None):

        self.text = text

        self.children = children if children else []

        # Create embedding
        self.embedding = embedding_model.encode(text)

    def is_leaf(self):

        return len(self.children) == 0


def summarize(text):

    prompt = f"""
You are summarizing documents.

Create a short semantic summary.

Text:

{text}

Summary:
"""

    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ]
    )

    return response.choices[0].message.content

def build_tree(nodes):

    # stop condition
    if len(nodes) == 1:

        return nodes[0]

    print(
        "Building level with nodes:",
        len(nodes)
    )

    embeddings = np.array(
        [
            node.embedding
            for node in nodes
        ]
    )

    # number of clusters
    n_clusters = max(
        1,
        len(nodes)//2
    )

    kmeans = KMeans(
        n_clusters=n_clusters,
        random_state=42,
        n_init="auto"
    )

    labels = kmeans.fit_predict(
        embeddings
    )

    new_nodes = []

    for cluster_id in range(n_clusters):

        cluster_nodes = []

        for index,node in enumerate(nodes):

            if labels[index] == cluster_id:

                cluster_nodes.append(node)

        combined_text = "\n".join(
            [
                node.text
                for node in cluster_nodes
            ]
        )

        summary = summarize(
            combined_text
        )

        parent = Node(
            summary,
            cluster_nodes
        )

        new_nodes.append(parent)

    return build_tree(new_nodes)

def retrieve(query, node):

    print("\nSearching:")
    print(node.text[:100])

    # reached original chunk

    if node.is_leaf():

        return node

    query_embedding = embedding_model.encode(
        query
    )

    scores = []

    for child in node.children:

        score = cosine_similarity(
            [query_embedding],
            [child.embedding]
        )[0][0]

        scores.append(score)

    best_index = np.argmax(
        scores
    )

    best_child = node.children[
        best_index
    ]

    return retrieve(
        query,
        best_child
    )

documents = [

"""
Transformers use attention mechanisms
to understand relationships between words.
""",


"""
Multi head attention allows transformers
to focus on different parts of input.
""",


"""
Positional encoding gives information
about word positions in a sequence.
""",


"""
CNNs are neural networks mainly used
for image processing.
""",


"""
ResNet introduced skip connections
to solve deep network problems.
""",

"""
DenseNet connects every layer
with previous layers.
"""

]

leaf_nodes = []


for doc in documents:

    leaf_nodes.append(
        Node(doc)
    )

print(
    "\nCreating RAPTOR tree..."
)

root = build_tree(
    leaf_nodes
)
print("\n")
print("="*60)
print("ROOT SUMMARY")
print("="*60)

print(root.text)

query = "Explain multi head attention"

result = retrieve(
    query,
    root
)

print("\n")
print("="*60)
print("FINAL RETRIEVED CHUNK")
print("="*60)

print(result.text)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8988.67it/s]



Creating RAPTOR tree...
Building level with nodes: 6
Building level with nodes: 3


ROOT SUMMARY
Positional encoding provides context for word positions in sequences, while CNNs, like ResNet and DenseNet, enhance image processing through skip connections and layer interconnectivity. Transformers leverage attention mechanisms, particularly multi-head attention, to understand relationships between words across different input segments.

Searching:
Positional encoding provides context for word positions in sequences, while CNNs, like ResNet and De

Searching:
Transformers utilize attention mechanisms to grasp word relationships, with multi-head attention ena

Searching:

Multi head attention allows transformers
to focus on different parts of input.



FINAL RETRIEVED CHUNK

Multi head attention allows transformers
to focus on different parts of input.

